In [2]:
import pandas as pd
import sqlite3

# -----------------------------
# 1. File URLs
# -----------------------------
meals_url = "https://assets.datacamp.com/production/repositories/4016/datasets/732c094b30a2e794d0b12b12547587a903126f68/meals.csv"
orders_url = "https://assets.datacamp.com/production/repositories/4016/datasets/606e6e9165c25477db078996fa7e0a3e994b93d3/orders.csv"
stock_url = "https://assets.datacamp.com/production/repositories/4016/datasets/10d9ad146a85010d836cfc93870aa464951f0640/stock.csv"

# -----------------------------
# 2. Define column names
# -----------------------------
meals_cols = ["meal_id", "eatery", "meal_price", "meal_cost"]
orders_cols = ["order_date", "user_id", "order_id", "meal_id", "order_quantity"]
stock_cols = ["stocking_date", "meal_id", "stocked_quantity"]

# -----------------------------
# 3. Read CSV files
#    header=None because the files do not include column names
# -----------------------------
meals = pd.read_csv(meals_url, header=None, names=meals_cols)
orders = pd.read_csv(orders_url, header=None, names=orders_cols)
stock = pd.read_csv(stock_url, header=None, names=stock_cols)

# Optional: convert date columns to datetime
orders["order_date"] = pd.to_datetime(orders["order_date"])
stock["stocking_date"] = pd.to_datetime(stock["stocking_date"])

# -----------------------------
# 4. Save local CSV copies (optional)
# -----------------------------
meals.to_csv("meals.csv", index=False)
orders.to_csv("orders.csv", index=False)
stock.to_csv("stock.csv", index=False)

# -----------------------------
# 5. Create SQLite database
# -----------------------------
conn = sqlite3.connect("delivr.db")

# Write tables into SQLite
meals.to_sql("meals", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)
stock.to_sql("stock", conn, if_exists="replace", index=False)

print("Database created: delivr.db")
print("Tables: meals, orders, stock")

Database created: delivr.db
Tables: meals, orders, stock


In [3]:
query = """
SELECT *
FROM meals
LIMIT 5;
"""

result = pd.read_sql_query(query, conn)
print(result)

   meal_id                    eatery  meal_price  meal_cost
0        0  'Leaning Tower of Pizza'         4.0       2.00
1        1  'Leaning Tower of Pizza'         3.5       1.25
2        2  'Leaning Tower of Pizza'         4.5       1.75
3        3  'Leaning Tower of Pizza'         4.0       0.75
4        4              'Burgatorio'         6.0       3.25


In [4]:
query = """
SELECT
    SUM(m.meal_price * o.order_quantity) AS revenue
FROM meals AS m
JOIN orders AS o
    ON m.meal_id = o.meal_id;
"""

result = pd.read_sql_query(query, conn)
print(result)

     revenue
0  260226.75
